!pip install ultralytics -q
!pip install supervision -q # Useful library for drawing seamless boxes

import torch
import cv2
import numpy as np
from ultralytics import YOLO
import os
from collections import deque
from google.colab import files
from tqdm.notebook import tqdm # For a nice progress bar

# Setup device parameters
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device.upper()}")
if device == 'cuda':
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

# Create output directory
output_dir = 'processed_videos'
os.makedirs(output_dir, exist_ok=True)
print("\nSetup Complete. Ready for video.")

In [1]:
!pip install ultralytics -q
!pip install supervision -q # Useful library for drawing seamless boxes

import torch
import cv2
import numpy as np
from ultralytics import YOLO
import os
from collections import deque
from google.colab import files
from tqdm.notebook import tqdm # For a nice progress bar

# Setup device parameters
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device.upper()}")
if device == 'cuda':
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

# Create output directory
output_dir = 'processed_videos'
os.makedirs(output_dir, exist_ok=True)
print("\nSetup Complete. Ready for video.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.4/212.4 kB 4.9 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using Device: CPU

Setup Complete. Ready for video.


In [2]:
# === CELL 2: Defining the Threat Analysis Logic ===

class ImpactAnalyzer:
    def __init__(self, history_length=15):
        """
        history_length: How many past frames to remember for calculating trends.
                        Higher = smoother but slower reaction time. 10-15 is good for 30fps video.
        """
        # Dictionary to store history of tracked objects.
        # Format: {track_id: deque([(box_coords, center_x, area), ...], maxlen=N)}
        self.track_history = {}
        self.history_length = history_length

    def get_box_center_area(self, box):
        # box format is usually [x1, y1, x2, y2]
        x1, y1, x2, y2 = box
        center_x = int((x1 + x2) / 2)
        center_y = int((y1 + y2) / 2)
        area = (x2 - x1) * (y2 - y1)
        return center_x, center_y, area

    def calculate_threat(self, track_id, current_box, frame_width):
        curr_cx, curr_cy, curr_area = self.get_box_center_area(current_box)
        frame_center_x = frame_width / 2

        # 1. Initialize history if new object
        if track_id not in self.track_history:
            self.track_history[track_id] = deque(maxlen=self.history_length)
            # We need at least a few frames to calculate a trend. Return safe for now.
            self.track_history[track_id].append((current_box, curr_cx, curr_area))
            return 0, (0, 255, 0) # Score 0, Green

        # Get past data (the oldest record in our history deque)
        past_box, past_cx, past_area = self.track_history[track_id][0]

        # --- THREAT Calculation Logic ---
        threat_score = 0

        # A. LOOMING FACTOR (Is it getting bigger?)
        # Calculate percentage growth compared to history
        if past_area > 0:
            growth_rate = (curr_area - past_area) / past_area
        else:
            growth_rate = 0

        # If growing significantly, increase threat score
        if growth_rate > 0.05: # >5% growth in history window (caution)
             threat_score += 30
        if growth_rate > 0.15: # >15% growth (danger looming fast)
             threat_score += 50

        # B. CENTERING FACTOR (Is it moving into my path?)
        # Calculate how far off-center it was vs is now
        past_offset = abs(past_cx - frame_center_x)
        curr_offset = abs(curr_cx - frame_center_x)

        # If it's getting closer to center AND it's already somewhat close
        if curr_offset < past_offset and curr_offset < (frame_width / 4):
             threat_score += 30

        # C. Ensure score is between 0 and 100
        threat_score = min(max(int(threat_score), 0), 100)

        # D. Determine Color
        if threat_score < 30:
            color = (0, 255, 0) # Green (Safe)
        elif threat_score < 70:
            color = (0, 255, 255) # Yellow (Caution - OpenCV uses BGR)
        else:
            color = (0, 0, 255) # Red (Danger)

        # Update history with current status for next loop
        self.track_history[track_id].append((current_box, curr_cx, curr_area))

        return threat_score, color

print("ImpactAnalyzer Class defined.")

ImpactAnalyzer Class defined.


In [ ]:
# === CELL 3: Upload Video ===
print("Please upload your video file (e.g., mp4, avi).")
uploaded = files.upload()
video_filename = next(iter(uploaded))
print(f"\nSelected: {video_filename}")

Please upload your video file (e.g., mp4, avi).


In [ ]:
# === CELL 4: Main Processing Loop ===

# --- Configurations ---
INPUT_VIDEO = video_filename
OUTPUT_VIDEO = os.path.join(output_dir, f"impact_analysis_{video_filename}")
MODEL_TYPE = 'yolov8s.pt' # Using 'small' model for better accuracy than 'nano' on T4 GPU
CONF_THRESHOLD = 0.4
# Vehicles only (Car, Motorcycle, Bus, Truck) in COCO dataset
TARGET_CLASSES = [2, 3, 5, 7]

# --- Initialize ---
print(f"Loading {MODEL_TYPE} to GPU...")
model = YOLO(MODEL_TYPE)
model.to(device)

# Initialize our custom threat analyzer brain
# We look back 10 frames to judge speed. Adjust based on your video FPS.
analyzer = ImpactAnalyzer(history_length=10)

# Open Video
cap = cv2.VideoCapture(INPUT_VIDEO)
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Video Writer setup
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

print(f"Starting processing on {total_frames} frames. This will take a moment...")

# --- Main Loop with Progress Bar ---
for _ in tqdm(range(total_frames), desc="Processing Frames"):
    success, frame = cap.read()
    if not success: break

    # 1. Run Tracking
    # persist=True is crucial to keep IDs the same between frames
    results = model.track(frame, persist=True, conf=CONF_THRESHOLD, classes=TARGET_CLASSES, verbose=False, device=device)

    # Get the boxes and track IDs from YOLO results
    # Handle cases where no objects are detected in a frame
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        track_ids = results[0].boxes.id.cpu().numpy().astype(int)
        classes = results[0].boxes.cls.cpu().numpy().astype(int)

        # 2. Loop through detected objects and analyze threat
        for box, track_id, cls_id in zip(boxes, track_ids, classes):
            x1, y1, x2, y2 = box

            # Calculate Threat score and get corresponding color
            score, color = analyzer.calculate_threat(track_id, box, width)

            # 3. Draw Custom Visuals (Drawing manually gives us more control than results.plot())
            # Draw Bounding Box
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # Create Label Text based on class name and threat score
            class_name = model.names[cls_id]
            label = f"ID:{track_id} {class_name} | Threat:{score}"

            # Draw label background rectangle for readability
            t_size = cv2.getTextSize(label, 0, 0.6, 2)[0]
            cv2.rectangle(frame, (x1, y1 - t_size[1] - 6), (x1 + t_size[0], y1), color, -1)

            # Draw label text (white text)
            cv2.putText(frame, label, (x1, y1 - 5), 0, 0.6, (255, 255, 255), 2)

    # Draw a reference center line on the frame (optional, helps visualize centering)
    cv2.line(frame, (int(width/2), height-50), (int(width/2), height), (200,200,200), 2)

    # Write frame to output video
    out.write(frame)

# Cleanup
cap.release()
out.release()
print("\nProcessing Complete.")

# Trigger download
print(f"Downloading processed video: {OUTPUT_VIDEO}")
files.download(OUTPUT_VIDEO)